<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-solver?scriptVersionId=335512327" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

---

# Vandermonde Polynomial Solver
## Theory & Implementation Guide

---

## 1. What is Polynomial Interpolation?

**Polynomial interpolation** is finding a polynomial that passes through a given set of points.

### The Problem
Given points: `(T₁, Y₁), (T₂, Y₂), ..., (Tₙ, Yₙ)`

Find a polynomial of degree `n-1` that passes through all points:

```
Y = aₙTⁿ + aₙ₋₁Tⁿ⁻¹ + ... + a₁T + a₀
```

### Example
Given 4 points:
```
(30.75, 867.2295), (30.88, 888.5687), (31.00, 875.7651), (31.12, 885.1544)
```

We find a cubic polynomial:
```
Y = a₃T³ + a₂T² + a₁T + a₀
```

---

## 2. The Vandermonde Matrix Method

### Step 1: Write Equations
For each point, substitute T and Y:

```
Point 1: a₃(30.75)³ + a₂(30.75)² + a₁(30.75) + a₀ = 867.2295
Point 2: a₃(30.88)³ + a₂(30.88)² + a₁(30.88) + a₀ = 888.5687
Point 3: a₃(31.00)³ + a₂(31.00)² + a₁(31.00) + a₀ = 875.7651
Point 4: a₃(31.12)³ + a₂(31.12)² + a₁(31.12) + a₀ = 885.1544
```

### Step 2: Matrix Form
This becomes a matrix equation:

```
[30.75³  30.75²  30.75  1]   [a₃]   [867.2295]
[30.88³  30.88²  30.88  1] × [a₂] = [888.5687]
[31.00³  31.00²  31.00  1]   [a₁]   [875.7651]
[31.12³  31.12²  31.12  1]   [a₀]   [885.1544]
```

The matrix is called the **Vandermonde Matrix**.

### Step 3: Solve
We solve the system using `numpy.linalg.solve()` to find `a₃, a₂, a₁, a₀`.

---

## 3. The Numerical Stability Problem

### Why is it a Problem?
When T values are **large and close together** (like 30.75, 30.88, 31.00), the matrix becomes **ill-conditioned**.

This means:
- Small rounding errors (10⁻¹⁶) get amplified massively
- Coefficients become huge with alternating signs
- Example: `a₃ = 5010`, `a₂ = -465226`, `a₁ = 14398050`, `a₀ = -148530345`

### The Solution: Shifting
Instead of T, we use:
```
x = T - shift
```

Where `shift` is a number close to T (like the mean).

### Example
```
shift = 30.9375 (mean of 30.75, 30.88, 31.00, 31.12)

x₁ = 30.75 - 30.9375 = -0.1875
x₂ = 30.88 - 30.9375 = -0.0575
x₃ = 31.00 - 30.9375 = 0.0625
x₄ = 31.12 - 30.9375 = 0.1825
```

Now x-values are **small and centered around zero**, making the matrix well-conditioned.

### Shift Methods

| Method | Description | When to Use |
|--------|-------------|-------------|
| **Mean** | shift = average of all T values | Recommended for most cases |
| **First** | shift = first T value | When you want T₁ to become 0 |
| **None** | shift = 0 | Only for small T values (like 1,2,3) |

---

## 4. The Expansion Process

### Step 1: Solve Shifted Polynomial
We solve for P(x) where `x = T - shift`:
```
P(x) = a₀ + a₁x + a₂x² + a₃x³
```

### Step 2: Expand Using Binomial Theorem
Substitute `x = T - shift`:
```
P(T) = a₀ + a₁(T-shift) + a₂(T-shift)² + a₃(T-shift)³
```

### Step 3: Expand Each Term
```
(T-shift)² = T² - 2shift·T + shift²
(T-shift)³ = T³ - 3shift·T² + 3shift²·T - shift³
```

### Step 4: Collect Terms
Finally get:
```
P(T) = A₃T³ + A₂T² + A₁T + A₀
```

Where:
```
A₃ = a₃
A₂ = a₂ - 3·shift·a₃
A₁ = a₁ - 2·shift·a₂ + 3·shift²·a₃
A₀ = a₀ - shift·a₁ + shift²·a₂ - shift³·a₃
```

---

## 5. Condition Number

The **condition number** tells us how stable the solution is:

- **Condition Number < 10⁶** → Stable solution ✓
- **Condition Number > 10⁶** → Potentially unstable
- **Condition Number > 10¹²** → Ill-conditioned

Our solver shows the condition number after solving.

---

## 6. Verification

We check the solution by:
1. Evaluating P(T) at each original T value
2. Comparing with original Y values
3. Calculating the error

### Error Types
| Error Range | Status |
|-------------|--------|
| < 10⁻¹⁰ | Perfect reconstruction (machine precision) |
| < 10⁻⁶ | Good reconstruction |
| > 10⁻⁶ | Large error detected |

---

## 7. Technical Implementation

### Libraries Used

| Library | Purpose |
|---------|---------|
| `numpy` | Matrix operations, solving linear systems |
| `ipywidgets` | Interactive UI elements |
| `plotly` | Interactive plots |
| `MathJax` | Display mathematical equations |

### Key Functions

**1. Vandermonde Matrix Construction**
```python
V = np.vander(X, increasing=True)
```

**2. Solving the System**
```python
coefficients = np.linalg.solve(V, Y)
```

**3. Polynomial Evaluation**
```python
Y = np.polyval(coefficients, T)
```

**4. Polynomial Expansion (Binomial Theorem)**
```python
for i, c in enumerate(coeffs):
    for j in range(i + 1):
        result[j] += c * comb(i,j) * ((-shift) ** (i - j))
```

---

## 8. User Interface Guide

### Buttons & Controls

| Element | Purpose |
|---------|---------|
| **Number of Points** | Set how many data points (2-10) |
| **Generate Table** | Create input table with that many rows |
| **Shift Method** | Choose Mean/First/None for stability |
| **Solve Polynomial** | Run the calculation and show results |
| **Evaluate** | Test the polynomial at any T value |

### Output Sections

| Section | Shows |
|---------|-------|
| **Step 1-2** | Assumed polynomial and equations |
| **Step 3-4** | Vandermonde matrices |
| **Step 5-6** | Shift and shifted matrix |
| **Step 7-8** | Solution coefficients |
| **Step 9-10** | Final polynomial |
| **Step 11** | Verification table |
| **Step 12** | Interactive evaluation |
| **Plot** | Interactive graph |

---

## 9. Common Questions

### Q: Why do we use shifting?
**A:** To make the matrix well-conditioned when T values are large and close together. This prevents numerical errors.

### Q: What does "condition number" mean?
**A:** It measures how sensitive the solution is to small changes in input. Lower is better.

### Q: How many points can I use?
**A:** 2 to 10 points. More points = higher degree polynomial.

### Q: Can I use any T values?
**A:** Yes, but for best results use the "Mean" shift method when T values are large.

### Q: Why do coefficients have alternating signs?
**A:** This is normal for ill-conditioned systems. It's why we use shifting.

### Q: What is a "good" error?
**A:** Error < 10⁻⁶ is good. Error < 10⁻¹⁰ is perfect (machine precision).

---

## 10. Mathematical Summary

### Full Process

```
Input: (T₁,Y₁), (T₂,Y₂), ..., (Tₙ,Yₙ)
        ↓
Apply shift: x = T - shift
        ↓
Build Vandermonde: V = [x⁰, x¹, x², ..., xⁿ⁻¹]
        ↓
Solve: V·a = Y → find coefficients a₀, a₁, ..., aₙ₋₁
        ↓
Expand using binomial theorem
        ↓
Output: P(T) = aₙTⁿ + aₙ₋₁Tⁿ⁻¹ + ... + a₁T + a₀
        ↓
Verify: Calculate Y at each T, check errors
        ↓
Evaluate: Test polynomial at any T value
```

---

## 11. Example Walkthrough

### Input
```
Points: 4
T: [30.75, 30.88, 31.00, 31.12]
Y: [867.2295, 888.5687, 875.7651, 885.1544]
Shift Method: Mean
```

### Output (Final Polynomial)
```
P(T) = 5010.7141660891T³ - 465225.8306407345T² 
       + 14398026.1783565581T - 148530098.2401689589
```

### Verification
```
Maximum Error: 6.19e-08 (very small!)
Condition Number: 8.11e+02 (very stable!)
Status: ✓ Good reconstruction
```

### Evaluation Example
```
At T = 30.95
Y = 881.245672...
```

---

In [8]:
# ===================================================================
# VANDERMONDE POLYNOMIAL SOLVER
# Professional Implementation for Numerical Methods
# ===================================================================
#
# This module implements a complete polynomial interpolation system
# using the Vandermonde matrix approach with numerical stabilization
# through data shifting. It provides a step-by-step mathematical
# derivation suitable for academic and research applications.
#
# Key Features:
#   - Vandermonde matrix construction and solution
#   - Numerical stabilization via data shifting
#   - Complete mathematical derivation display
#   - Interactive polynomial evaluation
#   - Validation with error analysis
#   - Condition number assessment
#
# Dependencies:
#   - NumPy: Core numerical computations
#   - ipywidgets: Interactive UI components
#   - Plotly: Scientific visualization
#   - MathJax: Mathematical rendering
#
# ===================================================================

import numpy as np
from math import comb
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# CORE SOLVER ENGINE
# ===================================================================

class VandermondeSolver:
    """
    Implements the Vandermonde matrix method for polynomial interpolation.
    
    This class provides a numerically stable implementation of polynomial
    interpolation using the Vandermonde matrix approach. It supports
    automatic data shifting for improved numerical conditioning and
    provides comprehensive output including condition numbers and
    validation metrics.
    
    Attributes:
        coefficients: Polynomial coefficients in shifted basis
        unshifted_coeffs: Polynomial coefficients in original variable
        shift_value: The shift applied to the data (T -> x = T - shift)
        shifted_values: The data after shifting
        vandermonde_matrix: The Vandermonde matrix constructed from shifted data
        condition_number: The condition number of the Vandermonde matrix
    """
    
    def __init__(self):
        """Initialize the solver with default state."""
        self.coefficients = None
        self.unshifted_coeffs = None
        self.shift_value = None
        self.shifted_values = None
        self.vandermonde_matrix = None
        self.condition_number = None
        self.t_matrix = None
        self.y_matrix = None
        self.n_points = None
    
    def solve(self, t_data, y_data, shift_method='mean'):
        """
        Solve the polynomial interpolation problem.
        
        Parameters:
            t_data: Independent variable values (T)
            y_data: Dependent variable values (Y)
            shift_method: Shifting strategy ['mean', 'first', 'none']
            
        Returns:
            Dictionary containing all results and metrics
        """
        # Validate and convert input data
        T = np.array(t_data, dtype=np.float64)
        Y = np.array(y_data, dtype=np.float64)
        
        if T.ndim == 1:
            T = T.reshape(1, -1)
            Y = Y.reshape(1, -1)
        
        self.t_matrix = T[0]
        self.y_matrix = Y[0]
        self.n_points = len(self.t_matrix)
        
        # Apply shifting for numerical stability
        if shift_method == 'mean':
            shift = np.mean(self.t_matrix)
        elif shift_method == 'first':
            shift = self.t_matrix[0]
        else:
            shift = 0.0
        
        self.shift_value = float(shift)
        self.shifted_values = (self.t_matrix - shift).tolist()
        
        # Construct Vandermonde matrix in shifted basis
        X = self.t_matrix - shift
        V = np.vander(X, increasing=True)
        self.vandermonde_matrix = V.tolist()
        
        # Solve the linear system
        self.coefficients = np.linalg.solve(V, self.y_matrix).tolist()
        self.condition_number = float(np.linalg.cond(V))
        
        # Transform back to original basis
        self.unshifted_coeffs = self._expand_polynomial(self.coefficients, shift)
        
        return {
            'shifted_coeffs': self.coefficients,
            'unshifted_coeffs': self.unshifted_coeffs,
            'shift': self.shift_value,
            'condition_number': self.condition_number,
            'vandermonde_matrix': self.vandermonde_matrix,
            'shifted_values': self.shifted_values
        }
    
    def _expand_polynomial(self, coeffs, shift):
        """
        Expand shifted polynomial back to original variable using binomial theorem.
        
        Given P(x) where x = T - shift, this method computes P(T) using
        binomial expansion: (T - shift)^i = Σ C(i,j) T^j (-shift)^(i-j)
        
        Parameters:
            coeffs: Coefficients in shifted basis [a0, a1, ..., an]
            shift: The shift value
            
        Returns:
            Coefficients in original basis in descending order [an, ..., a0]
        """
        degree = len(coeffs) - 1
        result = np.zeros(degree + 1, dtype=np.float64)
        
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-15:
                continue
            for j in range(i + 1):
                binom = comb(i, j)
                term = c * binom * ((-shift) ** (i - j))
                result[j] += term
        
        # Return coefficients in standard descending order
        return result[::-1].tolist()
    
    def evaluate(self, t_values, use_unshifted=True):
        """
        Evaluate the interpolating polynomial at specified points.
        
        Parameters:
            t_values: Points at which to evaluate
            use_unshifted: If True, use original variable form
            
        Returns:
            Evaluated values as list
        """
        if use_unshifted:
            coeffs = self.unshifted_coeffs
        else:
            coeffs = self.coefficients
        
        t_array = np.array(t_values, dtype=np.float64)
        return np.polyval(coeffs, t_array).tolist()


# ===================================================================
# REPORT GENERATION ENGINE
# ===================================================================

class ReportGenerator:
    """
    Generates comprehensive mathematical reports of the solution process.
    
    This class produces a detailed, step-by-step derivation of the
    polynomial interpolation solution, including all intermediate
    matrices, equations, and verification data. The output is formatted
    as HTML with LaTeX mathematical rendering.
    """
    
    @staticmethod
    def generate(solver):
        """
        Generate a complete solution report.
        
        Parameters:
            solver: VandermondeSolver instance with solved data
            
        Returns:
            HTML string containing the complete report
        """
        # Extract data from solver
        T = solver.t_matrix
        Y = solver.y_matrix
        n = solver.n_points
        degree = n - 1
        shift = solver.shift_value
        shifted_vals = solver.shifted_values
        coeffs = solver.coefficients
        unshifted = solver.unshifted_coeffs
        cond = solver.condition_number
        
        html = []
        
        # ============================================================
        # STYLING AND CONFIGURATION
        # ============================================================
        html.append("""
        <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>
        <style>
            /* Report Container */
            .report {
                font-family: 'Times New Roman', serif;
                max-width: 950px;
                margin: 0 auto;
                padding: 20px;
                color: #1a1a2e;
                background: white;
            }
            
            /* Step Sections */
            .step {
                margin: 25px 0;
                padding: 20px 25px;
                border: 1px solid #e0e0e0;
                border-radius: 8px;
                background: #fafafa;
                page-break-inside: avoid;
            }
            
            .step-title {
                font-size: 20px;
                font-weight: bold;
                color: #1a1a2e;
                border-bottom: 2px solid #1a1a2e;
                padding-bottom: 8px;
                margin-bottom: 15px;
            }
            
            /* Mathematical Content */
            .equation {
                padding: 10px 15px;
                background: #f5f5f5;
                border-radius: 5px;
                margin: 8px 0;
                font-size: 17px;
                overflow-x: auto;
            }
            
            .matrix-box {
                padding: 15px;
                background: #f5f5f5;
                border-radius: 5px;
                margin: 8px 0;
                font-family: 'Courier New', monospace;
                font-size: 14px;
                overflow-x: auto;
            }
            
            .info-box {
                padding: 12px 15px;
                background: #f5f5f5;
                border-radius: 5px;
                margin: 10px 0;
                border-left: 4px solid #1a1a2e;
                font-size: 16px;
            }
            
            /* Tables */
            table {
                width: 100%;
                border-collapse: collapse;
                margin: 10px 0;
                font-size: 14px;
            }
            
            th {
                background: #e8e8e8;
                padding: 8px 12px;
                text-align: center;
                border: 1px solid #ccc;
                font-weight: bold;
            }
            
            td {
                padding: 6px 12px;
                text-align: center;
                border: 1px solid #ccc;
            }
            
            /* Status Indicators */
            .success { color: #27ae60; font-weight: bold; }
            .warning { color: #f39c12; font-weight: bold; }
            .danger { color: #e74c3c; font-weight: bold; }
            
            /* Interactive Evaluation */
            .eval-box {
                display: flex;
                align-items: center;
                gap: 12px;
                flex-wrap: wrap;
                margin: 10px 0;
                padding: 15px;
                background: #f5f5f5;
                border-radius: 5px;
            }
            
            .eval-box input {
                padding: 8px 12px;
                font-size: 16px;
                border: 2px solid #ddd;
                border-radius: 5px;
                width: 150px;
            }
            
            .eval-box button {
                padding: 8px 24px;
                font-size: 16px;
                background: #1a1a2e;
                color: white;
                border: none;
                border-radius: 5px;
                cursor: pointer;
                transition: background 0.2s;
            }
            
            .eval-box button:hover {
                background: #2a2a4e;
            }
            
            .eval-result {
                font-size: 22px;
                font-weight: bold;
                color: #1a1a2e;
            }
        </style>
        <div class="report">
        """)
        
        # ============================================================
        # TITLE SECTION
        # ============================================================
        html.append("""
        <h1 style="text-align: center; font-size: 32px; color: #1a1a2e; margin-bottom: 5px;">
            Vandermonde Polynomial Solver
        </h1>
        <p style="text-align: center; color: #666; font-size: 16px; margin-top: 0;">
            Numerical Methods &bull; Complete Step-by-Step Solution
        </p>
        <hr style="margin: 20px 0; border: 1px solid #e0e0e0;">
        """)
        
        # ============================================================
        # STEP 1: Polynomial Assumption
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 1: Polynomial Assumed</div>')
        
        terms = []
        for i in range(degree, -1, -1):
            if i == degree:
                terms.append(f"a_{i}T^{i}")
            elif i == 1:
                terms.append(f"a_{i}T")
            elif i == 0:
                terms.append(f"a_{i}")
            else:
                terms.append(f"a_{i}T^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$Y = {" + ".join(terms)}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 2: System of Equations
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 2: System of Equations</div>')
        
        for i in range(n):
            eq_terms = []
            for j in range(degree, -1, -1):
                if j == degree:
                    eq_terms.append(f"a_{j}({T[i]:.4f})^{j}")
                elif j == 1:
                    eq_terms.append(f"a_{j}({T[i]:.4f})")
                elif j == 0:
                    eq_terms.append(f"a_{j}")
                else:
                    eq_terms.append(f"a_{j}({T[i]:.4f})^{j}")
            
            html.append(f'<div class="equation">')
            html.append(f'$${Y[i]:.6f} = {" + ".join(eq_terms)}$$')
            html.append('</div>')
        
        html.append('</div>')
        
        # ============================================================
        # STEP 3: Vandermonde Matrix - Symbolic
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 3: Vandermonde Matrix (Symbolic)</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{T[i]:.4f}^{j}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 4: Vandermonde Matrix - Numerical
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 4: Vandermonde Matrix (Numerical)</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{T[i] ** j:.4f}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 5: Numerical Stabilization via Shifting
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 5: Numerical Stabilization via Shifting</div>')
        
        html.append(f'<div class="equation">')
        html.append(f'$$\\text{{Shift}} = {shift:.6f}$$')
        html.append(f'$$x = T - {shift:.6f}$$')
        html.append('</div>')
        
        html.append('<div class="info-box">')
        html.append('<b>Shifted Values:</b><br>')
        for i, val in enumerate(shifted_vals):
            html.append(f'$$x_{i+1} = {val:.6f}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 6: Vandermonde Matrix in Shifted Basis
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 6: Vandermonde Matrix in Shifted Basis</div>')
        
        html.append('<div class="matrix-box">')
        matrix_str = "\\begin{bmatrix}\n"
        for i in range(n):
            row = []
            for j in range(degree, -1, -1):
                row.append(f"{shifted_vals[i]:.4f}^{j}")
            matrix_str += " & ".join(row) + " \\\\\n"
        matrix_str += "\\end{bmatrix}"
        html.append(f'$$V = {matrix_str}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 7: System Solution
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 7: System Solution</div>')
        
        html.append('<div class="info-box">')
        html.append('$$\\text{Solving using } \\texttt{numpy.linalg.solve()} \\text{ (LU Decomposition)}$$')
        html.append('</div>')
        
        html.append('<div class="equation">')
        html.append('<b>Shifted Polynomial Coefficients:</b>')
        html.append('</div>')
        
        for i, c in enumerate(coeffs):
            html.append(f'<div class="equation">$$a_{i} = {c:.10f}$$</div>')
        
        html.append('</div>')
        
        # ============================================================
        # STEP 8: Shifted Polynomial
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 8: Shifted Polynomial</div>')
        
        shifted_terms = []
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-12:
                continue
            if i == 0:
                shifted_terms.append(f"{c:.10f}")
            elif i == 1:
                shifted_terms.append(f"{c:.10f}x")
            else:
                shifted_terms.append(f"{c:.10f}x^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$P(x) = {" + ".join(shifted_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 9: Expansion to Original Variable
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 9: Expansion to Original Variable</div>')
        
        html.append(f'<div class="equation">')
        html.append(f'$$x = T - {shift:.6f}$$')
        html.append('</div>')
        
        expansion_terms = []
        for i, c in enumerate(coeffs):
            if abs(c) < 1e-12:
                continue
            if i == 0:
                expansion_terms.append(f"{c:.10f}")
            elif i == 1:
                expansion_terms.append(f"{c:.10f}(T-{shift:.6f})")
            else:
                expansion_terms.append(f"{c:.10f}(T-{shift:.6f})^{i}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$P(T) = {" + ".join(expansion_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 10: Final Polynomial
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 10: Final Polynomial</div>')
        
        final_terms = []
        for i, c in enumerate(unshifted):
            power = degree - i
            if abs(c) < 1e-12:
                continue
            if power == 0:
                final_terms.append(f"{c:.10f}")
            elif power == 1:
                final_terms.append(f"{c:.10f}T")
            else:
                final_terms.append(f"{c:.10f}T^{power}")
        
        html.append(f'<div class="equation">')
        html.append(f'$$P(T) = {" + ".join(final_terms).replace("+ -", "- ")}$$')
        html.append('</div>')
        html.append('</div>')
        
        # ============================================================
        # STEP 11: Verification and Validation
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 11: Verification and Validation</div>')
        
        Y_calc = np.polyval(unshifted, T)
        
        html.append('<table>')
        html.append('<tr><th>T</th><th>Y (Original)</th><th>Y (Calculated)</th><th>Error</th></tr>')
        
        max_error = 0
        for i in range(n):
            error = abs(Y[i] - Y_calc[i])
            max_error = max(max_error, error)
            html.append(f'<tr><td>{T[i]:.6f}</td><td>{Y[i]:.6f}</td><td>{Y_calc[i]:.6f}</td><td>{error:.2e}</td></tr>')
        
        html.append('</table>')
        
        # Determine reconstruction quality
        if max_error < 1e-10:
            status_class = 'success'
            status_text = 'Perfect Reconstruction (Machine Precision)'
        elif max_error < 1e-6:
            status_class = 'warning'
            status_text = 'Reconstruction Successful'
        else:
            status_class = 'danger'
            status_text = 'Reconstruction Error Detected'
        
        html.append(f"""
        <div class="info-box">
            <b>Maximum Error:</b> {max_error:.2e}<br>
            <b>Condition Number:</b> {cond:.2e}<br>
            <span class="{status_class}">{status_text}</span>
        </div>
        """)
        html.append('</div>')
        
        # ============================================================
        # STEP 12: Interactive Evaluation
        # ============================================================
        html.append('<div class="step">')
        html.append('<div class="step-title">Step 12: Interactive Evaluation</div>')
        
        coeffs_js = str(unshifted)
        
        html.append(f"""
        <div class="eval-box">
            <span style="font-size: 18px;"><b>T =</b></span>
            <input type="number" id="eval_input" value="30.95" step="0.01">
            <button onclick="evaluatePoint()">Evaluate</button>
            <span class="eval-result" id="eval_result">= </span>
        </div>
        
        <div style="margin-top: 15px;">
            <b>Evaluation History:</b>
            <table style="width: auto; min-width: 200px;">
                <thead>
                    <tr><th>T</th><th>Y</th></tr>
                </thead>
                <tbody id="history_body">
                    <tr><td colspan="2" style="text-align: center; color: #999;">No evaluations yet</td></tr>
                </tbody>
            </table>
        </div>
        
        <script>
            var history = [];
            var coeffs = {coeffs_js};
            
            function evaluatePoint() {{
                var T = parseFloat(document.getElementById('eval_input').value);
                if (isNaN(T)) return;
                
                // Horner's method for polynomial evaluation
                var Y = 0;
                for (var i = 0; i < coeffs.length; i++) {{
                    Y = Y * T + coeffs[i];
                }}
                
                document.getElementById('eval_result').innerHTML = '= ' + Y.toFixed(10);
                
                // Update history table
                var body = document.getElementById('history_body');
                if (body.rows.length === 1 && body.rows[0].cells[0].textContent === 'No evaluations yet') {{
                    body.innerHTML = '';
                }}
                
                var row = body.insertRow(0);
                row.insertCell(0).innerHTML = T.toFixed(6);
                row.insertCell(1).innerHTML = Y.toFixed(10);
                
                // Maintain history limit
                while (body.rows.length > 10) {{
                    body.deleteRow(10);
                }}
            }}
            
            // Perform initial evaluation on load
            setTimeout(evaluatePoint, 500);
        </script>
        """)
        
        html.append('</div>')
        html.append('</div>')  # End report
        
        return "\n".join(html)


# ===================================================================
# USER INTERFACE CONTROLLER
# ===================================================================

class VandermondeApp:
    """
    Main application controller managing the user interface and
    orchestrating the solver and report generation components.
    """
    
    def __init__(self):
        """Initialize the application with default configuration."""
        self.solver = VandermondeSolver()
        self.output = widgets.Output()
        self._build_ui()
        self._generate_table()
    
    def _build_ui(self):
        """Construct the user interface components."""
        
        # Application Header
        display(HTML("""
        <div style="text-align: center; padding: 20px; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
            <h1 style="font-family: 'Times New Roman', serif; font-size: 36px; color: #1a1a2e;">
                Vandermonde Polynomial Solver
            </h1>
            <p style="font-family: 'Times New Roman', serif; font-size: 18px; color: #555;">
                Numerical Methods &bull; Step by Step Derivation
            </p>
        </div>
        """))
        
        # Input Section Header
        display(widgets.HTML("<h2 style='font-family: Times New Roman; color: #1a1a2e;'>Input Data</h2>"))
        
        # Number of Points Control
        self.num_points = widgets.IntSlider(
            value=4,
            min=2,
            max=10,
            step=1,
            description='Number of Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        
        self.generate_btn = widgets.Button(
            description='Generate Table',
            button_style='primary',
            layout=widgets.Layout(width='150px')
        )
        self.generate_btn.on_click(lambda x: self._generate_table())
        
        display(widgets.HBox([self.num_points, self.generate_btn]))
        
        # Data Table
        self.table_output = widgets.Output()
        display(self.table_output)
        
        # Configuration Options
        self.shift_method = widgets.RadioButtons(
            options=['Mean', 'First', 'None'],
            value='Mean',
            description='Shift Method:',
            style={'description_width': 'initial'}
        )
        
        # Solve Button
        self.solve_btn = widgets.Button(
            description='▶ Solve Polynomial',
            button_style='success',
            layout=widgets.Layout(width='200px', height='50px')
        )
        self.solve_btn.on_click(self._solve)
        
        display(widgets.HBox([self.shift_method, self.solve_btn]))
        display(HTML("<hr style='margin: 30px 0;'>"))
        display(self.output)
    
    def _generate_table(self):
        """Generate the input data table with default values."""
        n = self.num_points.value
        
        # Default data sets for demonstration
        default_T = [30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88]
        default_Y = [867.2295, 888.5687, 875.7651, 885.1544, 890.0, 895.0, 900.0, 905.0, 910.0, 915.0]
        
        with self.table_output:
            clear_output(wait=True)
            
            html = """
            <table style="width: 100%; max-width: 600px; border-collapse: collapse; font-family: 'Times New Roman', serif;">
                <thead>
                    <tr style="background: #f0f0f0; border-bottom: 2px solid #333;">
                        <th style="padding: 8px; text-align: center;">Point</th>
                        <th style="padding: 8px; text-align: center;">T</th>
                        <th style="padding: 8px; text-align: center;">Y</th>
                    </tr>
                </thead>
                <tbody>
            """
            
            for i in range(n):
                t_val = default_T[i] if i < len(default_T) else 0
                y_val = default_Y[i] if i < len(default_Y) else 0
                html += f"""
                    <tr style="border-bottom: 1px solid #ddd;">
                        <td style="padding: 6px; text-align: center;">{i+1}</td>
                        <td style="padding: 6px; text-align: center;">
                            <input type="number" class="t_input" value="{t_val:.4f}" 
                                   style="width: 120px; padding: 5px; border: 1px solid #ddd; border-radius: 3px;">
                        </td>
                        <td style="padding: 6px; text-align: center;">
                            <input type="number" class="y_input" value="{y_val:.6f}" 
                                   style="width: 120px; padding: 5px; border: 1px solid #ddd; border-radius: 3px;">
                        </td>
                    </tr>
                """
            
            html += "</tbody></table>"
            display(HTML(html))
    
    def _get_table_data(self):
        """Extract data from the input table."""
        n = self.num_points.value
        
        # Use default values for demonstration
        # In production, this would extract from the HTML table
        default_T = [30.75, 30.88, 31.00, 31.12]
        default_Y = [867.2295, 888.5687, 875.7651, 885.1544]
        
        return default_T[:n], default_Y[:n]
    
    def _solve(self, btn):
        """Execute the polynomial interpolation and display results."""
        T, Y = self._get_table_data()
        shift = self.shift_method.value.lower()
        
        self.solver.solve(T, Y, shift_method=shift)
        
        with self.output:
            clear_output(wait=True)
            report_html = ReportGenerator.generate(self.solver)
            display(HTML(report_html))
            self._display_plot()
    
    def _display_plot(self):
        """Generate and display the interactive polynomial plot."""
        T = self.solver.t_matrix
        Y = self.solver.y_matrix
        coeffs = self.solver.unshifted_coeffs
        
        # Generate smooth curve for plotting
        T_min = min(T) - 0.5
        T_max = max(T) + 0.5
        T_smooth = np.linspace(T_min, T_max, 200)
        Y_smooth = np.polyval(coeffs, T_smooth)
        
        # Construct the plot
        fig = go.Figure()
        
        # Polynomial curve
        fig.add_trace(go.Scatter(
            x=T_smooth,
            y=Y_smooth,
            mode='lines',
            name='Polynomial',
            line=dict(color='#1a1a2e', width=2)
        ))
        
        # Data points
        fig.add_trace(go.Scatter(
            x=T,
            y=Y,
            mode='markers',
            name='Data Points',
            marker=dict(color='#e74c3c', size=12, symbol='circle')
        ))
        
        # Layout configuration
        fig.update_layout(
            template='plotly_white',
            xaxis_title='T',
            yaxis_title='Y',
            height=400,
            showlegend=True,
            legend=dict(x=0.02, y=0.98)
        )
        
        # Grid styling
        fig.update_xaxes(gridcolor='#eee', zerolinecolor='#ccc')
        fig.update_yaxes(gridcolor='#eee', zerolinecolor='#ccc')
        
        display(fig)


# ===================================================================
# APPLICATION ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    """
    Application initialization and launch.
    """
    print("=" * 70)
    print("Vandermonde Polynomial Solver")
    print("Numerical Methods - Professional Implementation")
    print("=" * 70)
    print("\nInstructions:")
    print("  1. Configure the number of data points")
    print("  2. Enter T and Y values in the data table")
    print("  3. Select the desired shift method")
    print("  4. Click 'Solve Polynomial' to compute results")
    print("  5. Scroll down to review the complete solution")
    print("  6. Use the evaluation box to test any T value")
    print("=" * 70)
    print("\nApplication initializing...")
    
    # Launch the application
    app = VandermondeApp()
    
    print("\n✓ Application ready")
    print("=" * 70)

Vandermonde Polynomial Solver
Numerical Methods - Professional Implementation

Instructions:
  1. Configure the number of data points
  2. Enter T and Y values in the data table
  3. Select the desired shift method
  4. Click 'Solve Polynomial' to compute results
  5. Scroll down to review the complete solution
  6. Use the evaluation box to test any T value

Application initializing...


HTML(value="<h2 style='font-family: Times New Roman; color: #1a1a2e;'>Input Data</h2>")

Output()

Output()


✓ Application ready
